# Conditional Flow Matching

In this lab session, you will implement a simple conditional flow matching model for a toy 2D problem.

## Dataset formatting

Before delving into the implementation, you will need to prepare data to feed to the neural network during training. We will rely on a well-known toy dataset: the 2 Moons dataset.

In [1]:
import os

os.environ["KERAS_BACKEND"] = "torch"

In [2]:
from sklearn.datasets import make_moons
import numpy as np

n_samples = 20000
noise = 0.05
moons, _ = make_moons(n_samples=n_samples, noise=noise)
moons = moons.astype(np.float32)
perm = np.random.permutation(len(moons))
split = int(0.9 * len(moons))

# Training set
train_x = moons[perm[:split]]
# Validation set
val_x = moons[perm[split:]]

In [3]:
import matplotlib.pyplot as plt

plt.figure(figsize=(4,4))
plt.scatter(train_x[:,0], train_x[:,1], s=2, alpha=0.4)
plt.axis('equal')
plt.show()

Now, recall that in conditional flow matching, we need to train a neural network to predict the velocity field by minimizing the following loss function:

\begin{equation}
\mathcal{L}(\theta) = \mathbb{E}_{t \sim \mathcal{U}(0, 1), \mathbf{x}_0 \sim p_0, \mathbf{x}_1 \sim p_{\text{data}}} \left[ \| \mathbf{v}_\theta(\mathbf{x}, t) - \mathbf{v}_t \|^2_2 \right]
\end{equation}

where 
\begin{equation}
\mathbf{x} = (1 - t) \mathbf{x}_0 + t \mathbf{x}_1
\end{equation}
 
and 
\begin{equation}
\mathbf{v}_t = \mathbf{x}_1 - \mathbf{x}_0
\end{equation}

is the target velocity field.

We will hence train our neural network $\mathbf{v}_\theta$ on samples of the form $(\mathbf{x}, t)$ as inputs and $\mathbf{v}_t$ as targets.

**Question 1.** Implement a custom data generator that will yield batches of samples during training.
In `keras`, this can be achieved by defining a class that inherits from the `Sequence` class:

In [ ]:
from keras.utils import Sequence

class MoonsSequence(Sequence):
    def __init__(self, x, batch_size=256):
        self.x = x
        self.batch_size = batch_size
        self.n = len(x)

    def __len__(self):
        return int(np.ceil(self.n / self.batch_size))

    def __getitem__(self, idx):
        start = idx * self.batch_size
        end = min(start + self.batch_size, self.n)
        # TODO: prepare xt, t, and target_v here
        return {'x': xt, 't': t}, target_v

batch_size = 256
train_seq = MoonsSequence(train_x, batch_size=batch_size)
val_seq = MoonsSequence(val_x, batch_size=batch_size)

## Velocity field model

For our velocity field model $\mathbf{v}_\theta$, we will use a simple feedforward neural network with fully connected layers.
This network will take as input the concatenation of the 2D point $\mathbf{x}$ and the time $t$, and will output a 2D vector representing the velocity at that point and time.

**Question 2.** Implement the velocity field model by filling the code below. As you can see, we cannot use the `Sequential` model of `keras` here, since we need to process two different types of inputs (the 2D point and the time). Instead, we will inherit from the `Model` class from `keras`. The `__init__` method is given to you and you will need to implement the `call` method that defines the forward pass of the model (i.e. computes the output of the model from `x` and `t` given as inputs).

In [ ]:
from keras.models import Model, Sequential
from keras.layers import Dense, Concatenate

class VelocityField(Model):
    def __init__(self, hidden_size=256):
        super().__init__()
        self.net = Sequential([
            Dense(units=hidden_size, activation="relu"),
            Dense(units=2)
        ])
        self.concatenate = Concatenate()
    
    def call(self, inputs):
        # Accept either a dict {'x': x, 't': t} or a tuple (x, t)
        if isinstance(inputs, dict):
            x = inputs['x']
            t = inputs['t']
        else:
            x, t = inputs
        # TODO: here
        return v_t_pred


**Question 3.** Train your model for 200 epochs using the Adam optimizer with a learning rate of 0.001 and visualize training and validation losses.

## Data generation

Now that you have a trained velocity field model, you can use it to generate new samples from the learned distribution.
To do so, we will implement the Euler-Maruyama method to solve the ODE defined by the learned velocity field.
This involves:
1. sampling initial points $\mathbf{x}_0$ from our prior distribution $p_0$,
2. iteratively updating the position of particles based on the predicted velocity field at each time step, up to time $t=1$:
\begin{equation}
\mathbf{x}_{t+\Delta t} = \mathbf{x}_t + \Delta t \cdot \mathbf{v}_\theta(\mathbf{x}_t, t) 
\end{equation}

**Question 4.** Fill in the blanks below to implement the data generation process using a trained velocity field model.

In [ ]:
def sample_flow_keras(model, n_samples=1000, n_steps=200):
    # TODO
    pass

In [ ]:
samples = sample_flow_keras(model, n_samples=5000, n_steps=1000)

plt.figure(figsize=(5,5))
plt.scatter(train_x[:,0], train_x[:,1], s=2, alpha=0.4, label='data')
plt.scatter(samples[:,0], samples[:,1], s=2, alpha=0.4, label='generated')
plt.legend()
plt.axis('equal')
plt.show()

## Improvements to the base model

The data generator you get from the previous section is quite basic and can be improved in several ways.

A first way to improve on that model would be tu encode time $t$ in a richer way. Fourier features have been shown to be effective in that regard.

**Question 5.** Using the code below, implement Fourier feature encoding for time $t$ in your velocity field model, retrain your model and generate new samples. Compare the results with the base model.

In [ ]:
from keras.layers import Layer
from keras import ops
import math

class FourierTime(Layer):
    def __init__(self, K=16, **kwargs):
        super().__init__(**kwargs)
        self.K = K
        freqs = 2 ** ops.arange(self.K, dtype=np.float32)
        self.freqs = freqs.reshape((1, self.K))

    def call(self, t):
        angles = 2 * math.pi * t * self.freqs
        s = ops.sin(angles)
        c = ops.cos(angles)
        return ops.concatenate((t, s, c), axis=-1)

In [ ]:
# Update your VelocityField class here

In [ ]:
model = VelocityField()

opt = Adam(learning_rate=1e-3)
model.compile(optimizer=opt, loss='mse')
history = model.fit(train_seq, validation_data=val_seq, epochs=200)

samples = sample_flow_keras(model, n_samples=5000, n_steps=1000)

plt.figure(figsize=(5,5))
plt.scatter(train_x[:,0], train_x[:,1], s=2, alpha=0.4, label='data')
plt.scatter(samples[:,0], samples[:,1], s=2, alpha=0.4, label='generated')
plt.legend()
plt.axis('equal')
plt.show()

**Question 6.** You might have seen that the score for the last epoch was not the best one. Set up early stopping such that the best model is recovered after the last iteration.